In [1]:
!pip install plotly


In [14]:

import pandas as pd
import numpy as np
import re
import requests
from scipy import stats
from datetime import datetime
import zipfile
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import scipy.stats as stats
from scipy.optimize import curve_fit
import shutil
from matplotlib import font_manager, rc
import subprocess
import sys
import os
import random
import glob
import time 
import folium
from folium.plugins import HeatMap
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm



In [15]:
# csv파일을 받아서 데이터 프레임으로 저장
# (출처 :https://data.seoul.go.kr/dataList/OA-12914/S/1/datasetView.do)
# CSV 파일 경로와 이름(승하차 데이터가 있는 csv 파일, 다운 받은 경로로 변경)
file_path = "/home/addd/Downloads/CARD_SUBWAY_MONTH_202312.csv"

df= pd.read_csv(file_path,header=0)
df = df.drop('등록일자', axis=1)

# CSV 파일 경로와 이름(경도와 위도에 대한 정보가 들어 있는 csv, 다운 받은 경로로 변경)
file_path = "/home/addd/Downloads/INTERN_YUSEONGMIN/GIS/finally_super_important_seoul_subway_data_the_end.csv"

df_1= pd.read_csv(file_path)


In [16]:
# csv파일 그냥 받으면 열이 1칸 씩 밀려서, 인덱스를 새로운 열로 추가
df.reset_index(inplace=True)

df.rename(columns={
    '사용일자': '노선명',
    '노선명': '역명',
    '역명': '승차총승객수',
    '승차총승객수': '하차총승객수',
    '하차총승객수': 'next_column'  # 'next_column'은 전송 날짜 이므로 쓰레기 데이터
}, inplace=True)
df.index.name = '사용일자'
df

,index,노선명,역명,승차총승객수,하차총승객수,next_column
사용일자,,,,,,
0,20231201,2호선,대림(구로구청),25995,26770,20231204
1,20231201,2호선,신도림,55447,55057,20231204
2,20231201,2호선,문래,25818,26561,20231204
3,20231201,2호선,영등포구청,24517,25006,20231204
4,20231201,2호선,당산,20266,23832,20231204
...,...,...,...,...,...,...
18951,20231231,중앙선,중랑,3423,3132,20240103
18952,20231231,중앙선,상봉(시외버스터미널),3665,3370,20240103
18953,20231231,중앙선,망우,5053,4834,20240103


In [17]:
# 괄호가 있으면 위도, 경도가 안 받아지므로 df의 '역명' 열을 순회하면서 처리합니다.
for i in df.index:
    station_name = df.loc[i, '역명']
    if "(" in station_name and ")" in station_name:
        # 괄호 밖과 안의 내용을 분리합니다.
        outside_brackets, inside_brackets = re.search(r'([^(]*)\(([^)]*)\)', station_name).groups()
        
        # 괄호 밖과 안의 내용이 df_1의 'name' 열에 존재하는지 확인합니다.
        outside_exists = outside_brackets.strip() in df_1['name'].values
        inside_exists = inside_brackets.strip() in df_1['name'].values
        
        # 괄호 밖의 내용이 df_1의 'name' 열에 존재하면, 해당 행의 역 이름을 그 단어로 변경합니다.
        if outside_exists:
            df.loc[i, '역명'] = outside_brackets.strip()
        # 괄호 안의 내용이 df_1의 'name' 열에 존재하면, 해당 행의 역 이름을 그 단어로 변경합니다.
        elif inside_exists:
            df.loc[i, '역명'] = inside_brackets.strip()


In [18]:
# 나중에 데이터 처리하기 편하게 미리 문자열로 변경
df['노선명'] = df['노선명'].astype(str)
df['역명'] = df['역명'].astype(str)
df_1['line'] = df_1['line'].astype(str)
df_1['name'] = df_1['name'].astype(str)

In [25]:
# 두 데이터프레임 합치기
df_merged = pd.merge(df_1, df, left_on=['name'], right_on=['역명'], how='right')
df_merged = df_merged.drop_duplicates(subset=['승차총승객수', '하차총승객수', 'next_column'])

In [26]:
# 병합 된 파일에서 위도와 경도가 NULL인 역명을 리스트로 저장(없어야 하나 원래 23개 쯤 나왔음, 서울 지하철 위경도 데이터 라벨링 하신 분이 빠트리거나 그 사이 신설된 역들들)
station_list = df_merged[df_merged['lat'].isna()].drop_duplicates(subset='역명')['역명'].tolist()
print(station_list)

[]


In [27]:
# 날짜 다른 애들끼리 합치기 위해서 '노선명', '역명', 'lat', 'lng', 'index'가 같은 행들끼리 그룹을 만들어 '승차총승객수'와 '하차총승객수'를 합칩니다.
df_merged = df_merged.groupby(['노선명', '역명', 'lat', 'lng']).sum().reset_index()

# '승차총승객수'와 '하차총승객수'를 합친 '총승객수' 컬럼을 생성합니다.
df_merged['총승객수'] = df_merged['승차총승객수'] + df_merged['하차총승객수']

In [24]:
'''# 위도와 경도가 NULL인 역들의 데이터를 Open Street Map 사용해서 구함(이미 구했음으로 주석 처리리)

from geopy.geocoders import Nominatim

# Nominatim 객체를 생성합니다.
geolocator = Nominatim(user_agent="geoapiExercises")

# 위도와 경도를 저장할 빈 리스트를 생성합니다.
location_list = []

# station_list의 각 역에 대해 위도와 경도를 구합니다.
for station in station_list:
    time.sleep(12)
    # 역의 이름에 '역'을 추가하고 그 위치를 구합니다.
    location = geolocator.geocode(station + '역')

    # 위치 정보가 있는 경우 위도와 경도를 리스트에 추가합니다.
    if location:
        location_list.append([station, location.latitude, location.longitude])

# df_1의 마지막 인덱스를 구합니다.
last_index = df_1.index[-1]

# location_list의 각 위치에 대해 새로운 행을 df_1에 추가합니다.
for i in range(len(location_list)):
    df_1.loc[last_index + i + 1] = {'name': location_list[i][0], 'lat': location_list[i][1], 'lng': location_list[i][2]}

# 잘 들어가졌나 체크

nan_rows = df_1[df_1['lat'].isna() | df_1['lng'].isna()]
print(nan_rows)
'''

      line name  code  lat  lng     index  노선명   역명  승차총승객수  하차총승객수  \
9362   NaN  NaN   NaN  NaN  NaN  20231216  경원선   연천    6268    6642   
9707   NaN  NaN   NaN  NaN  NaN  20231216  경원선   전곡    2546    2150   
9710   NaN  NaN   NaN  NaN  NaN  20231216  경원선  초성리     383     353   
9995   NaN  NaN   NaN  NaN  NaN  20231217  경원선  초성리     150     144   
9996   NaN  NaN   NaN  NaN  NaN  20231217  경원선   전곡    1196    1153   
9997   NaN  NaN   NaN  NaN  NaN  20231217  경원선   연천    2123    2092   
10779  NaN  NaN   NaN  NaN  NaN  20231218  경원선  초성리      99      92   
10780  NaN  NaN   NaN  NaN  NaN  20231218  경원선   전곡    1066     898   
10781  NaN  NaN   NaN  NaN  NaN  20231218  경원선   연천    1491    1476   
11271  NaN  NaN   NaN  NaN  NaN  20231219  경원선  초성리     106     108   
11272  NaN  NaN   NaN  NaN  NaN  20231219  경원선   전곡    1250    1088   
11273  NaN  NaN   NaN  NaN  NaN  20231219  경원선   연천    1712    1665   
11678  NaN  NaN   NaN  NaN  NaN  20231220  경원선   연천    1225    1252   
11679 

In [28]:
# 노선명 1호선 부터 경의 중앙선 까지 따로 뽑고.
line_names = df_merged['노선명'].unique()

# 각 노선명에 대해 무작위 색상을 부여합니다.
color_dict = {line: '#%06X' % random.randint(0, 0xFFFFFF) for line in line_names}

In [39]:
# 인구를 히트맵으로 나타내기
import folium
from folium.plugins import HeatMap
import random

m = folium.Map(location=[37.5665, 126.9780], zoom_start=2)

for i in range(0,len(df_merged)):
   popup_text = "노선명: {}<br>역명: {}<br>승차총승객수: {}<br>하차총승객수: {}".format(df_merged.iloc[i]['노선명'], df_merged.iloc[i]['역명'], df_merged.iloc[i]['승차총승객수'], df_merged.iloc[i]['하차총승객수'])
   folium.CircleMarker(
      location=[df_merged.iloc[i]['lat'], df_merged.iloc[i]['lng']],
      radius=float(df_merged.iloc[i]['승차총승객수'])/55000, 
      color=color_dict[df_merged.iloc[i]['노선명']], # 노선명에 따른 색상을 설정합니다.
      fill=True,
      fill_color=color_dict[df_merged.iloc[i]['노선명']], # 노선명에 따른 색상을 설정합니다.
      popup=folium.Popup(folium.IFrame(html=popup_text, width=200, height=100)) # 역의 정보를 원에 표시합니다.
   ).add_to(m)

# 지도를 로컬에 HTML 파일로 저장합니다.
m.save('map_heatmap.html')

# 파일의 경로를 출력합니다.
import os
print(os.path.abspath('map_heatmap.html'))


/home/addd/Downloads/INTERN_YUSEONGMIN/GIS/map_heatmap.html


In [40]:
import plotly.express as px
import random

# 유니크한 노선명을 찾습니다.
line_names = df_merged['노선명'].unique()

# 더 진한 색상을 위해 색상 코드 범위를 조정합니다.
available_colors = ['#%06X' % random.randint(0, 0x7FFFFF) for _ in range(len(line_names))]
color_dict = dict(zip(line_names, random.sample(available_colors, len(line_names))))

# df_merged 데이터를 이용하여 지도를 그립니다.
fig = px.scatter_mapbox(df_merged, lat='lat', lon='lng',
                        color=df_merged['노선명'].apply(lambda x: color_dict[x]),
                        size='승차총승객수',
                        color_continuous_scale=px.colors.cyclical.IceFire,
                        size_max=15, zoom=10,
                        hover_name='노선명',  # 마우스 오버시 보여질 주요 텍스트
                        hover_data=['역명'])  # 추가 정보로 '역명'을 포함

fig.update_layout(mapbox_style="open-street-map")
fig.show()


In [44]:
import re
import random
import folium
import os
import pandas as pd

# ... 이전 코드 ...

m = folium.Map(location=[37.5665, 126.9780], zoom_start=10)

# 전체 '총승객수'의 평균을 계산합니다.
avg_total_passenger = df_merged['총승객수'].mean()

# 원의 크기를 정할 상수입니다.
size_factor = 8  # 원의 크기를 조절합니다.(너무 작으면 올립니다.)

for i in range(0, len(df_merged)):
    popup_text = "노선명: {}<br>역명: {}<br>승차총승객수: {}<br>하차총승객수: {}<br>총승객수: {}".format(
        df_merged.iloc[i]['노선명'],
        df_merged.iloc[i]['역명'],
        df_merged.iloc[i]['승차총승객수'],
        df_merged.iloc[i]['하차총승객수'],
        df_merged.iloc[i]['총승객수']
    )
    
    # 원의 크기를 '총승객수'에 따라 상대적으로 조정합니다.
    radius = (float(df_merged.iloc[i]['총승객수']) / avg_total_passenger) * size_factor

    # CircleMarker를 생성하고 맵에 추가합니다.
    folium.CircleMarker(
        location=[df_merged.iloc[i]['lat'], df_merged.iloc[i]['lng']],
        radius=radius,
        color=color_dict[df_merged.iloc[i]['노선명']], 
        fill=True,
        fill_color=color_dict[df_merged.iloc[i]['노선명']],
        fill_opacity=0.3,  # 원의 색상 투명도를 70%로 설정합니다.
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

# 지도를 로컬에 HTML 파일로 저장합니다.
m.save('map.html')

# 파일의 경로를 출력합니다.
print(os.path.abspath('map.html'))


/home/addd/Downloads/INTERN_YUSEONGMIN/GIS/map.html


In [31]:
# csv파일로 저장하기기
df_merged.to_csv('df_merged_202312.csv', index=False)
df_1.to_csv('202312_update_subway_geo.csv', index=False)